# Import of packages and arguments needed for data visualization

In [1]:
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter, MaxNLocator, MultipleLocator,  AutoMinorLocator, FixedLocator
import matplotlib.lines as mlines
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from collections import defaultdict
import numpy as np
import os
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "mathtext.fontset": "stix",
    "font.serif": ["STIXGeneral"],
})

import importlib
import utils
from logger import ExperimentLogger

importlib.reload(utils)
argFile = 'parameters.json'
try:
    del args
except NameError:
    pass
args = utils.loadJson(argFile)    
    
def get_path_to_sim(dataset):
    if dataset == "synthetic":
        path_to_sim = os.path.join(args.path_to_res_folder, f"run_K={args.K}_M={args.M}.npy")
    elif dataset == "MNIST":
        path_to_sim = os.path.join(args.path_to_res_folder, f"run_MNIST.npy")
    elif dataset == "CIFAR":
        path_to_sim = os.path.join(args.path_to_res_folder, f"run_CIFAR.npy")
    elif dataset == "fMNIST":
        path_to_sim = os.path.join(args.path_to_res_folder, f"run_fMNIST.npy")
    return path_to_sim

path_to_sim = os.path.join(args.path_to_res_folder, f"run_K={args.K}_M={args.M}.npy")
path_to_sim_ODEs = os.path.join(args.path_to_res_folder, f"ODES_K={args.K}_M={args.M}.npy")

configs = {
    "standard":  {"colors" : ["#FA7070", "#F8BFD4FF"], "marker" : "*", "marker_size" : 8},
    "Sco-standard":  {"colors" :["#FAC20C", "#EBD17E"], "marker" : "s", "marker_size" : 6},
    "LoRA":         {"colors" : ["#1D4ED8", "#93C5FD"], "marker" : "D", "marker_size" : 6},
    "Sco-LoRA":     {"colors" :["#15803D", "#86EFAC"], "marker" : "o", "marker_size" : 6},
}


# Some checks to read the saved files

In [2]:
def check_keys_inside_file(runs=False):
    # 2. Extract and display unique parameters
    if not runs:
        print("No runs found in this file or file doesn't exist.")
    else:
        print(f"Total number of logged runs: {len(runs)}\n")
        print("--- UNIQUE PARAMETERS INSIDE THE FILE ---")

        # Grab all metadata keys present in the first run to build our search list
        metadata_keys = runs[0]["metadata"].keys()

        for key in metadata_keys:
            # Use a set comprehension to gather every unique value seen across runs
            unique_values = {run["metadata"].get(key) for run in runs if key in run["metadata"]}

            # Sort if possible for clean reading (falls back to unsorted if types mismatch)
            try:
                sorted_values = sorted(list(unique_values))
            except TypeError:
                sorted_values = list(unique_values)

            print(f"  * {key:<12} : {sorted_values}")
            
def check_match_in_keys(runs=False):
    if not runs:
        print("The file is completely empty or doesn't exist.")
    else:
        meta = runs[0]["metadata"]
        print("--- LIVE PARAMETER DEBUGGER ---")
        print(f"{'Parameter':<12} | {'Inside File (meta)':<20} | {'Current Script (args)':<20} | Match?")
        print("-" * 65)

        # Helper to check type-agnostic matches
        def check(v1, v2, is_float=False):
            if is_float:
                return np.isclose(float(v1), float(v2))
            if str(v1).lower() in ['true', 'false'] or str(v2).lower() in ['true', 'false']:
                b1 = str(v1).lower() in ['true', '1']
                b2 = str(v2).lower() in ['true', '1']
                return b1 == b2
            return str(v1) == str(v2)

        # Test every single constraint from your match function
        print(f"{'L':<12} | {str(meta.get('L')):<20} | {str(getattr(args, 'L', 'MISSING')):<20} | {check(meta.get('L'), getattr(args, 'L', None))}")
        print(f"{'gamma':<12} | {str(meta.get('gamma')):<20} | {str(getattr(args, 'gamma', 'MISSING')):<20} | {check(meta.get('gamma'), getattr(args, 'gamma', None), True)}")
        print(f"{'method':<12} | {str(meta.get('method')):<20} | {str(getattr(args, 'method', 'MISSING')):<20} | {check(meta.get('method'), getattr(args, 'method', None))}")
        print(f"{'rho':<12} | {str(meta.get('rho')):<20} | {str(getattr(args, 'rho', 'MISSING')):<20} | {check(meta.get('rho'), getattr(args, 'rho', None), True)}")
        print(f"{'M':<12} | {str(meta.get('M')):<20} | {str(getattr(args, 'M', 'MISSING')):<20} | {check(meta.get('M'), getattr(args, 'M', None))}")
        print(f"{'K':<12} | {str(meta.get('K')):<20} | {str(getattr(args, 'K', 'MISSING')):<20} | {check(meta.get('K'), getattr(args, 'K', None))}")
        print(f"{'N':<12} | {str(meta.get('N')):<20} | {str(getattr(args, 'N', 'MISSING')):<20} | {check(meta.get('N'), getattr(args, 'N', None))}")
        print(f"{'alpha':<12} | {str(meta.get('alpha')):<20} | {str(getattr(args, 'alpha', 'MISSING')):<20} | {check(meta.get('alpha'), getattr(args, 'alpha', None), True)}")
          
path_to_sim = get_path_to_sim(args.dataset)
runs = ExperimentLogger.load_group(path_to_sim)
runs_ODEs = ExperimentLogger.load_group(path_to_sim_ODEs)

print("checking for sims :")  
check_keys_inside_file(runs)
print("checking for ODEs")
check_keys_inside_file(runs_ODEs)

print("checking for sims :")  
check_match_in_keys(runs)
print("checking for ODEs")
check_match_in_keys(runs_ODEs)

checking for sims :
Total number of logged runs: 2

--- UNIQUE PARAMETERS INSIDE THE FILE ---
  * N            : [1000]
  * M            : [5]
  * K            : [10]
  * rho          : [0.5]
  * dataset      : ['synthetic']
  * method       : ['LoRA', 'standard']
  * scoring_method : ['SDGM']
  * rows_to_freeze : [5]
  * informed_initialization : [False]
  * alpha        : [50]
  * gamma        : [1]
  * L            : [5]
  * alpha_W      : [0.5]
  * alpha_H      : [0.5]
  * alpha_a      : [0.5]
  * alpha_b      : [0.5]
  * teacher_initialization : ['committee']
  * teacher_hl_initialization : ['ungraded']
  * teacher_activation_function : ['erf']
  * student_activation_function : ['erf']
  * student_initialization : ['unspecializing']
  * seed         : [1001]
checking for ODEs
Total number of logged runs: 2

--- UNIQUE PARAMETERS INSIDE THE FILE ---
  * N            : [1000]
  * M            : [5]
  * K            : [10]
  * rho          : [0.5]
  * dataset      : ['synthetic']
  *

# Import the data from past runs with constraints

In [3]:
def metadata_matches(meta, args, synthetic=True, rho=None, method=None, scoring_method=None):
    rho = args.rho if rho is None else rho
    method = args.method if method is None else method
    scoring_method = getattr(args, "scoring_method", None) if scoring_method is None else scoring_method

    checks = {
        "method": (meta.get("method"), method),
        "rho": (float(meta.get("rho", 0.0)), float(rho)),
        "K": (int(meta.get("K", 0)), int(args.K)),
        "M": (int(meta.get("M", 0)), int(args.M)),
        #"gamma": (float(meta["gamma"]), float(args.gamma)),
        #"activation_function": (
        #    str(meta["activation_function"]),
        #    str(args.activation_function),
        #),
        #"informed_initialization": (
        #    str(meta["informed_initialization"]),
        #    str(args.informed_initialization)
        #),
    }
    if synthetic:
        checks["alpha"] = (float(meta.get("alpha", 0.0)), float(args.alpha))
        checks["N"] = (int(meta.get("N", 0)), int(args.N))
    # Conditionally check scoring method if the method utilizes scoring mechanisms
    if method in ["Sco-LoRA", "Sco-standard"] and scoring_method is not None:
        checks["scoring_method"] = (str(meta.get("scoring_method", "")), str(scoring_method))

    if method not in ["Sco-standard", "standard"]:
        checks["L"] = (int(meta.get("L", 0)), int(args.L))

    ok = True

    for key, (found, expected) in checks.items():
        if key == "rho":
            match = np.isclose(found, expected)
        else:
            match = found == expected

        if not match:
            # print(f"Mismatch for '{key}': found {found}, expected {expected}")
            ok = False
            
    return ok

def load_runs_with_constraints(path, args, rho=None):
    runs = ExperimentLogger.load_group(path)

    filtered = [
        r for r in runs
        if metadata_matches(r["metadata"], args,rho)
    ]

    if len(filtered) == 0:
        print(f"No matching configurations found in {os.path.basename(path)}!")
        return None

    if len(filtered) > 1:
        print(f"{len(filtered)} runs found in {os.path.basename(path)}!")
        for i in range(len(filtered)):
            print(filtered[i]["metadata"])
        return [filtered[0]]
    
    return filtered

def select_run_by_scoring(runs, args, method, scoring_method):
    """Filters trajectory runs matching both method and scoring_method."""
    original_scoring = getattr(args, "scoring_method", None)
    args.scoring_method = scoring_method
    try:
        for r in runs:
            meta = r.get("metadata", {})
            if metadata_matches(meta, args, method=method):
                return r
    finally:
        args.scoring_method = original_scoring
    return None

def runs_to_dict(path,args,rho=None):
    """Load runs with constraints and convert them into a dictionary format suitable for analysis.
    Input:
        path (str): The path to the file containing the runs.
        args (Namespace): The arguments containing the constraints.
        rho (float, optional): The value of rho to match.
    Output:
        dict: A dictionary containing the metadata and logs of the runs, organized by keys.
    """
    runs = load_runs_with_constraints(path, args, rho)
    #print("runs", runs)
    if runs == None :
        return None
    logs = runs[0]["logs"]

    out = {"metadata": runs[0]["metadata"]}

    # global steps if consistent, otherwise keep per-key
    out["steps"] = {}

    for key, entry in logs.items():
        out["steps"][key] = np.array(entry["steps"])

        values = entry["values"]

        if len(values) == 0:
            continue

        out[key] = np.array(values, dtype=object)

    return out
        
# Load the simulation
path_to_sim = get_path_to_sim(args.dataset)
path_to_sim_ODEs = os.path.join(args.path_to_res_folder, f"ODES_K={args.K}_M={args.M}.npy")
print(path_to_sim)
df = runs_to_dict(path_to_sim, args)
df_ODES = runs_to_dict(path_to_sim_ODEs, args)


#print(df["metadata"])
#print(df_ODES["metadata"])

/home/theom/continual_learning/results/run_K=10_M=5.npy


# Helpful functions for transfer and forgetting

In [4]:
def extract_metric_at_switch(run, metric_key, P, ODES=False, integration_step=None):
    logs = run["logs"]
    if metric_key not in logs:
        return None

    values = np.array(logs[metric_key]["values"], dtype=float)
    if len(values) == 0:
        return None
    
    if not ODES:
        steps = np.array(logs[metric_key]["steps"])
        idx = np.argmin(np.abs(steps - P))
        return float(values[idx])
    else:
        N = int(run["metadata"]["N"])
        x_axis = (np.arange(len(values)) * integration_step * N)
        idx = np.argmin(np.abs(x_axis - P))
        return float(values[idx])


def extract_metric(run, metric_key, mode="last", ODES=False, integration_step=None):
    logs = run["logs"]
    if metric_key not in logs:
        return None

    values = np.asarray(logs[metric_key]["values"], dtype=float)
    if len(values) == 0:
        return None

    if mode == "last":
        return float(values[-1])
    elif mode == "full":
        meta = run["metadata"]
        P = float(meta["alpha"]) * int(meta["N"])
        if ODES:
            N = int(meta["N"])
            x = np.arange(len(values)) * integration_step * N
        else:
            x = np.asarray(logs[metric_key]["steps"])
        idx = len(values) // 2
        return values[idx:]
    else:
        raise ValueError("mode must be 'last' or 'full'")


def compute_metric(run, metric_type, ODES=False, integration_step=None, extract_mode="last"):
    meta = run["metadata"]
    alpha = float(meta["alpha"])
    N = int(meta["N"])
    P = alpha * N

    switch_loss_1 = extract_metric_at_switch(run, "test_loss_1", P, ODES=ODES, integration_step=integration_step)
    switch_loss_2 = extract_metric_at_switch(run, "test_loss_2", P, ODES=ODES, integration_step=integration_step)

    loss_1 = extract_metric(run, "test_loss_1", extract_mode)
    loss_2 = extract_metric(run, "test_loss_2", extract_mode)

    if (switch_loss_1 is None or switch_loss_2 is None or loss_1 is None or loss_2 is None):
        return None

    if metric_type == "forgetting":
        metric_1 = (np.log10(loss_1) - np.log10(switch_loss_1))
        metric_2 = (np.log10(loss_2) - np.log10(switch_loss_2))
    elif metric_type == "transfer":
        metric_1 = (np.log10(switch_loss_1) - np.log10(loss_1))
        metric_2 = (np.log10(switch_loss_2) - np.log10(loss_2))
    else:
        raise ValueError("metric_type must be 'forgetting' or 'transfer'")

    return metric_1, metric_2


def aggregate_metric(runs, metric_type, task=1, extract_mode="last", ODES=False, integration_step=None,):
    data = []

    for run in runs or []:
        rho = float(run["metadata"]["rho"])

        metrics = compute_metric(run, metric_type, extract_mode=extract_mode, ODES=ODES, integration_step=integration_step,)

        if metrics is None:
            continue

        metric_1, metric_2 = metrics
        value = (metric_1 if task == 1 else metric_2)

        data.append((rho, value))
        
    data.sort(key=lambda x: x[0])

    if len(data) == 0:
        return np.array([]), []
    
    rhos = np.array([x[0] for x in data])

    if extract_mode == "last":
        values = np.array([x[1] for x in data])
    elif extract_mode == "full":
        values = [x[1] for x in data]
    else:
        raise ValueError("extract_mode must be 'last' or 'full'")

    return rhos, values

# Delete runs in the logs if needed

In [5]:
import numpy as np
import os

def delete_runs_with_args(path, args):
    """
    Deletes runs from the .npy file that match the provided args.
    """
    def match_predicate(meta):
        return (
            int(meta.get("L", 0)) == int(args.L) and
            #float(meta.get("gamma", 0)) == float(args.gamma) and
            str(meta.get("method", "")) == str(args.method) and
            np.isclose(float(meta.get("rho", 0)), float(args.rho)) 
            #int(meta.get("M", 0)) == int(args.M) and
            #int(meta.get("K", 0)) == int(args.K) and
            #int(meta.get("N", 0)) == int(args.N) and
            #float(meta.get("alpha", 0)) == float(args.alpha) and
            #int(meta.get("seed", 0)) == int(args.seed)
        )

    ExperimentLogger.delete_runs(path, match_predicate)

path_to_sim = os.path.join(args.path_to_res_folder, f"ODES_K={args.K}_M={args.M}.npy")
args.L=8
args.rho=0.5
args.method="Sco-LoRA"
Ls=[1,2,3,4,5,6,7,8,9,10]
#for L in Ls:
#    args.L = L
#    delete_runs_with_args(path_to_sim, args)